In [1]:
import os
import json

# Create the data directory if it doesn't exist
os.makedirs('data', exist_ok=True)

# Create a sample documents.json file
sample_data = [
    {"id": "doc1", "title": "Introduction to Python", "content": "Python is a high-level programming language known for its readability and versatility."},
    {"id": "doc2", "title": "Vector Space Model", "content": "The vector space model is an algebraic model for representing text documents as vectors of identifiers."},
    {"id": "doc3", "title": "TF-IDF Explained", "content": "TF-IDF stands for Term Frequency-Inverse Document Frequency, a numerical statistic that reflects how important a word is to a document."},
    {"id": "doc4", "title": "Machine Learning Basics", "content": "Machine learning involves algorithms that improve automatically through experience and by the use of data."},
    {"id": "doc5", "title": "Natural Language Processing", "content": "NLP is a subfield of linguistics, computer science, and artificial intelligence concerned with the interactions between computers and human language."}
]

with open('data/documents.json', 'w', encoding='utf-8') as f:
    json.dump(sample_data, f, indent=4)

print("✅ Sample dataset created at 'data/documents.json'")

✅ Sample dataset created at 'data/documents.json'


In [2]:
import os
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
# ─────────────────────────────────────────
# 1. LOAD DOCUMENTS
# ─────────────────────────────────────────
def load_documents(filepath: str) -> tuple[list[str], list[str]]:
    """Load documents from a JSON file.
    Returns a tuple of (doc_ids, doc_texts).
    """
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    doc_ids = [item["id"] for item in data]
    doc_texts = [item["title"] + " " + item["content"] for item in data]
    return doc_ids, doc_texts, data
# ─────────────────────────────────────────
# 2. BUILD TF-IDF MATRIX
# ─────────────────────────────────────────
def build_tfidf_matrix(documents: list[str]):
    """
    Vectorize the documents using TF-IDF.

    TF (Term Frequency)     = (# times word appears in doc) / (total words in doc)
    IDF (Inverse Doc Freq)  = log(total docs / docs containing word)
    TF-IDF                  = TF × IDF

    Returns the vectorizer and the TF-IDF matrix.
    """
    vectorizer = TfidfVectorizer(
        stop_words="english",       # Remove common English stop words
        lowercase=True,             # Normalize to lowercase
        ngram_range=(1, 2),         # Use unigrams AND bigrams for better matching
        max_df=0.85,                # Ignore terms that appear in >85% of docs (too common)
        min_df=1,                   # Minimum 1 document must contain the term
    )

    tfidf_matrix = vectorizer.fit_transform(documents)
    return vectorizer, tfidf_matrix
# ─────────────────────────────────────────
# 3. SEARCH FUNCTION
# ─────────────────────────────────────────
def search(query: str, vectorizer, tfidf_matrix, doc_ids: list, data: list, top_k: int = 5):
    """
    Given a user query, return the top_k most relevant documents.

    Steps:
    1. Transform the query using the SAME vectorizer (important!)
    2. Compute cosine similarity between query vector and all doc vectors
    3. Sort by similarity (descending) and return top_k results

    Cosine Similarity = (A · B) / (||A|| × ||B||)
    Range: 0 (no match) to 1 (perfect match)
    """
    # Step 1: Vectorize the query
    query_vector = vectorizer.transform([query])

    # Step 2: Compute cosine similarity
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # Step 3: Get indices sorted by descending similarity
    ranked_indices = np.argsort(similarities)[::-1]

    # Step 4: Collect top_k results (only those with similarity > 0)
    results = []
    for idx in ranked_indices[:top_k]:
        score = similarities[idx]
        if score > 0:
            results.append({
                "rank": len(results) + 1,
                "id": doc_ids[idx],
                "title": data[idx]["title"],
                "score": round(float(score), 4),
                "snippet": data[idx]["content"][:150] + "..."
            })

    return results
# ─────────────────────────────────────────
# 4. DISPLAY RESULTS
# ─────────────────────────────────────────
def display_results(query: str, results: list):
    """Pretty-print the search results."""
    print("\n" + "=" * 65)
    print(f"  🔍  Search Results for: \"{query}\"")
    print("=" * 65)

    if not results:
        print("  No matching documents found.")
    else:
        for r in results:
            print(f"\n  Rank #{r['rank']}  |  Score: {r['score']}")
            print(f"  📄 [{r['id']}] {r['title']}")
            print(f"  {r['snippet']}")
            print("  " + "-" * 60)

    print()
    # ─────────────────────────────────────────
# 5. MAIN PROGRAM
# ─────────────────────────────────────────
def main():
    # Determine dataset path relative to this script
    base_dir = os.getcwd()
    data_path = os.path.join(base_dir, "data", "documents.json") # Changed '../data' to 'data'

    print("\n╔════════════════════════════════════════════╗")
    print("║   VSM Search Engine  —  TF-IDF Edition     ║")
    print("╚════════════════════════════════════════════╝\n")

    # Load documents
    print("📂 Loading documents...")
    doc_ids, doc_texts, data = load_documents(data_path)
    print(f"   ✅ Loaded {len(doc_texts)} documents.\n")

    # Build TF-IDF matrix
    print("🧮 Building TF-IDF matrix...")
    vectorizer, tfidf_matrix = build_tfidf_matrix(doc_texts)
    print(f"   ✅ Matrix shape: {tfidf_matrix.shape}  "
          f"(docs × unique terms)\n")

    # Interactive search loop
    print("💡 Type your search query. Type 'quit' to exit.\n")
    while True:
        query = input("🔎 Enter query: ").strip()
        if query.lower() in ("quit", "exit", "q"):
            print("\n👋 Goodbye!\n")
            break
        if not query:
            continue

        results = search(query, vectorizer, tfidf_matrix, doc_ids, data, top_k=5)
        display_results(query, results)


if __name__ == "__main__":
    main()



╔════════════════════════════════════════════╗
║   VSM Search Engine  —  TF-IDF Edition     ║
╚════════════════════════════════════════════╝

📂 Loading documents...
   ✅ Loaded 5 documents.

🧮 Building TF-IDF matrix...
   ✅ Matrix shape: (5, 112)  (docs × unique terms)

💡 Type your search query. Type 'quit' to exit.

🔎 Enter query: NLP

  🔍  Search Results for: "NLP"

  Rank #1  |  Score: 0.1838
  📄 [doc5] Natural Language Processing
  NLP is a subfield of linguistics, computer science, and artificial intelligence concerned with the interactions between computers and human language....
  ------------------------------------------------------------

🔎 Enter query: LEARNING

  🔍  Search Results for: "LEARNING"

  Rank #1  |  Score: 0.3714
  📄 [doc4] Machine Learning Basics
  Machine learning involves algorithms that improve automatically through experience and by the use of data....
  ------------------------------------------------------------

🔎 Enter query: QUIT

👋 Goodbye!

